In [1]:
# =============================================================
#  BANK MARKETING PROJECT (Machine Learning)
# =============================================================

In [2]:
# ==============================
# 1. INSTALL DEPENDENCIES
# ==============================

!pip install -q lightgbm catboost imbalanced-learn xgboost scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.2 MB/s eta 0:00:00


In [3]:
# ==============================
# 2. IMPORTS LIBRARIES
# ==============================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, matthews_corrcoef, cohen_kappa_score, confusion_matrix)

from imblearn.metrics import geometric_mean_score
from imblearn.over_sampling import ADASYN

In [4]:
# ==============================
# 3. LOAD DATASET
# ==============================

df = pd.read_csv('/content/bank-additional-full (1).csv', sep=';')

In [5]:
# ==============================
# 4. PREPROCESSING
# ==============================

# Drop Leakage Feature
df = df.drop('duration', axis=1)

# Handle "unknown" Values
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])

# Encode Target
df['y'] = df['y'].map({'no': 0, 'yes': 1})

# One-Hot Encoding
df = pd.get_dummies(df, drop_first=True)

In [6]:
# ==============================
# 5. SPLIT DATA
# ==============================

# Split Features & Target
X = df.drop('y', axis=1)
y = df['y']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ADASYN Oversampling
adasyn = ADASYN(random_state=42)
X_train_ad, y_train_ad = adasyn.fit_resample(X_train_scaled, y_train)

In [7]:
# =========================================
# 6. CUSTOM METRICS: SPECIFICITY & G-MEAN
# =========================================
def specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

def gmean(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn)
    spec = tn / (tn + fp)
    return np.sqrt(sensitivity * spec)

In [8]:
# ==============================
# 7. EVALUATION METRICS
# ==============================
from sklearn.metrics import make_scorer

scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'specificity': make_scorer(specificity),
    'f1': 'f1',
    'roc_auc': 'roc_auc',
    'mcc': make_scorer(matthews_corrcoef),
    'g_mean': make_scorer(gmean),
    'kappa': make_scorer(cohen_kappa_score)
}

In [9]:
# ==============================
# 8. METRICS FUNCTION
# ==============================

def compute_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    print(f"Accuracy    : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision   : {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall      : {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Specificity : {tn / (tn + fp):.4f}")
    print(f"F1 Score    : {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"ROC-AUC     : {roc_auc_score(y_true, y_prob):.4f}")
    print(f"MCC         : {matthews_corrcoef(y_true, y_pred):.4f}")
    print(f"G-Mean      : {geometric_mean_score(y_true, y_pred):.4f}")
    print(f"Kappa       : {cohen_kappa_score(y_true, y_pred):.4f}")

In [10]:
# ==============================
# 9. CROSS VALIDATION
# ==============================

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

In [11]:
# ==============================
# MODEL: LogisticRegression
# ==============================
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight=None
)

params = {
    "C": [0.0001, 0.001, 0.01,0.1],
    "penalty": ["l1", "l2"],
    "solver": ["liblinear", "saga"]
}

rscv = RandomizedSearchCV(
    model,
    param_distributions=params,
    n_iter=10,
    scoring=scoring,
    refit='f1',
    cv=skf,
    n_jobs=-1
)

rscv.fit(X_train_ad, y_train_ad)

best =rscv.best_estimator_

y_pred = best.predict(X_test_scaled)
y_prob = best.predict_proba(X_test_scaled)[:,1]

print("Model: Logistic Regression")
compute_metrics(y_test, y_pred, y_prob)

print("\nBest Parameters:", rscv.best_params_)

Model: Logistic Regression
Accuracy    : 0.7355
Precision   : 0.2615
Recall      : 0.7392
Specificity : 0.7350
F1 Score    : 0.3864
ROC-AUC     : 0.7968
MCC         : 0.3219
G-Mean      : 0.7371
Kappa       : 0.2639

Best Parameters: {'solver': 'liblinear', 'penalty': 'l2', 'C': 0.0001}


In [12]:
# ==============================
# MODEL: Decision Tree
# ==============================
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier()

params = {
    "max_depth":[5,10,20,30],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4]
}

rscv = RandomizedSearchCV(
    model,
    param_distributions=params,
    n_iter=20,
    scoring=scoring,
    refit='roc_auc',
    cv=skf,
    n_jobs=-1
)

rscv.fit(X_train_ad, y_train_ad)



y_pred = best.predict(X_test_scaled)
y_prob = best.predict_proba(X_test_scaled)[:,1]

print("Model: Decision Tree")
compute_metrics(y_test, y_pred, y_prob)

print("\nBest Parameters:", rscv.best_params_)

Model: Decision Tree
Accuracy    : 0.7355
Precision   : 0.2615
Recall      : 0.7392
Specificity : 0.7350
F1 Score    : 0.3864
ROC-AUC     : 0.7968
MCC         : 0.3219
G-Mean      : 0.7371
Kappa       : 0.2639

Best Parameters: {'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': 20}


In [13]:
# ==============================
# MODEL: Gradient Boosting
# ==============================
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier()

params = {
    "n_estimators":[200,300],
    "learning_rate":[0.03,0.05,0.1]
}

rscv = RandomizedSearchCV(
    model,
    param_distributions=params,
    n_iter=10,
    scoring=scoring,
    refit='roc_auc',
    cv=skf,
    n_jobs=-1
)

rscv.fit(X_train_ad, y_train_ad)



y_pred = best.predict(X_test_scaled)
y_prob = best.predict_proba(X_test_scaled)[:,1]

print("Model: Gradient Boosting")
compute_metrics(y_test, y_pred, y_prob)

print("\nBest Parameters:", rscv.best_params_)

Model: Gradient Boosting
Accuracy    : 0.7355
Precision   : 0.2615
Recall      : 0.7392
Specificity : 0.7350
F1 Score    : 0.3864
ROC-AUC     : 0.7968
MCC         : 0.3219
G-Mean      : 0.7371
Kappa       : 0.2639

Best Parameters: {'n_estimators': 300, 'learning_rate': 0.1}


In [14]:
# ==============================
# MODEL: Random Forest
# ==============================
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(class_weight='balanced', n_jobs=-1,random_state=42)

params = {
    "n_estimators":[100,200],
    "max_depth":[10,15,20],
    "min_samples_split":[5,10],
    "min_samples_leaf":[2,4],
    "max_features":["sqrt"],
    "criterion":["entropy"]

}

rscv = RandomizedSearchCV(
    model,
    param_distributions=params,
    n_iter=10,
    scoring=scoring,
    refit='f1',
    cv=skf,
    n_jobs=-1
)

rscv.fit(X_train_ad, y_train_ad)

best = rscv.best_estimator_


y_pred = best.predict(X_test_scaled)
y_prob = best.predict_proba(X_test_scaled)[:,1]

print("Model: Random Forest")
compute_metrics(y_test, y_pred, y_prob)

print("\nBest Parameters:", rscv.best_params_)

Model: Random Forest
Accuracy    : 0.8926
Precision   : 0.5256
Recall      : 0.4763
Specificity : 0.9454
F1 Score    : 0.4997
ROC-AUC     : 0.8018
MCC         : 0.4404
G-Mean      : 0.6710
Kappa       : 0.4397

Best Parameters: {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 20, 'criterion': 'entropy'}


In [23]:
# ==============================
# MODEL: XGBoost
# ==============================
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import ADASYN
from xgboost import XGBClassifier


pipeline = Pipeline([
    ('adasyn', ADASYN(sampling_strategy=0.5, random_state=42)),
    ('model', XGBClassifier(eval_metric='logloss', use_label_encoder=False))
])

params = {
    "model__n_estimators":[300,500,700],
    "model__learning_rate":[0.01,0.03,0.05],
    "model__max_depth":[3,4,5],
    "model__subsample":[0.8,1.0],
    "model__colsample_bytree":[0.8,1.0],
    "model__gamma":[0,1],
    "model__reg_lambda":[1,2]
}

model = RandomizedSearchCV(pipeline, params, n_iter=3, cv=skf, scoring=scoring, refit='f1', n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("Model: XGBoost")
compute_metrics(y_test, y_pred, y_prob)

Model: XGBoost
Accuracy    : 0.8880
Precision   : 0.5029
Recall      : 0.4688
Specificity : 0.9412
F1 Score    : 0.4852
ROC-AUC     : 0.7963
MCC         : 0.4228
G-Mean      : 0.6642
Kappa       : 0.4224


In [24]:
# ==============================
# MODEL: LightGBM
# ==============================
from imblearn.pipeline import pipeline
from imblearn.over_sampling import ADASYN
from lightgbm import LGBMClassifier

pipe = Pipeline([
    ('adasyn', ADASYN(sampling_strategy=0.5, random_state=42)),
    ('model', LGBMClassifier(class_weight='balanced', verbosity=-1))
])

params = {
    "model__n_estimators":[300,500],
    "model__learning_rate":[0.03,0.05],
    "model__num_leaves":[31,63,127]
}

model = RandomizedSearchCV(pipe, params, n_iter=3, cv=skf, scoring=scoring, refit='recall', n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("Model: LightGBM")
compute_metrics(y_test, y_pred, y_prob)

Model: LightGBM
Accuracy    : 0.8786
Precision   : 0.4683
Recall      : 0.5722
Specificity : 0.9175
F1 Score    : 0.5150
ROC-AUC     : 0.7998
MCC         : 0.4494
G-Mean      : 0.7246
Kappa       : 0.4464


In [16]:
# ==============================
# MODEL: MLP
# ==============================


from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RandomizedSearchCV

mlp = MLPClassifier(
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=3,
    random_state=42
)

params_mlp = {
    "hidden_layer_sizes": [(16), (32)],
    "activation": ["relu"],
    "alpha": [1.0,5.0,10.0,20.0],
    "learning_rate_init": [1e-4],
    "batch_size": [32, 64],
    "solver": ["adam"],

}

rscv_mlp = RandomizedSearchCV(
    mlp,
    param_distributions=params_mlp,
    n_iter=12,
    scoring='precision',
    cv=3,
    n_jobs=-1,
    random_state=42
)

rscv_mlp.fit(X_train_ad, y_train_ad)

best_mlp = rscv_mlp.best_estimator_

y_prob_mlp = best_mlp.predict_proba(X_test_scaled)[:, 1]
y_pred_mlp = best_mlp.predict(X_test_scaled)
print("Model: MLP")
compute_metrics(y_test, y_pred_mlp, y_prob_mlp)

Model: MLP
Accuracy    : 0.7840
Precision   : 0.3026
Recall      : 0.7026
Specificity : 0.7944
F1 Score    : 0.4230
ROC-AUC     : 0.7997
MCC         : 0.3575
G-Mean      : 0.7471
Kappa       : 0.3151


In [27]:
# ==============================
# MODEL: Bagging
# ==============================
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV


base_tree = DecisionTreeClassifier(
    max_depth=None,
    class_weight='balanced',
    random_state=42
)

bagging = BaggingClassifier(
    estimator=base_tree,
    n_estimators=500,
    n_jobs=-1,
    random_state=42
)

params_bag = {
    "max_samples": [0.8, 1.0],
    "max_features": [0.8, 0.9],
    "estimator__min_samples_split": [2, 5],
    "estimator__min_samples_leaf": [1, 2]
}

rscv_bag = RandomizedSearchCV(
    bagging,
    param_distributions=params_bag,
    n_iter=10,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

rscv_bag.fit(X_train_ad, y_train_ad)
best_bag = rscv_bag.best_estimator_


y_pred_bag = best_bag.predict(X_test_scaled)
y_prob_bag = best_bag.predict_proba(X_test_scaled)[:, 1]

print("Model: Bagging ")
compute_metrics(y_test, y_pred_bag, y_prob_bag)



Fitting 3 folds for each of 10 candidates, totalling 30 fits
Model: Bagging 
Accuracy    : 0.8927
Precision   : 0.5368
Recall      : 0.3459
Specificity : 0.9621
F1 Score    : 0.4207
ROC-AUC     : 0.7835
MCC         : 0.3753
G-Mean      : 0.5769
Kappa       : 0.3646
